# Punchline Mesh — Live Studio (thin launcher)

Mounts the `punchline-mesh-src` dataset, starts the Streamlit humor studio with a real Gemma (measured surprise/resolution/efficiency/bad-surprise off the logits), and exposes it publicly through a Cloudflare quick tunnel. The `*.trycloudflare.com` URL prints below and is announced to ntfy.sh while the session runs (~8h).

Theory + code: see THEORY.md / WRITEUP.md inside the mounted source.

In [ ]:
import glob, os, re, shutil, subprocess, sys, time, urllib.request

# --- locate mounted source (dataset zip is auto-extracted under /kaggle/input) ---
apps = glob.glob('/kaggle/input/**/app.py', recursive=True)
assert apps, 'punchline-mesh-src dataset not attached'
SRC = os.path.dirname(apps[0])
DST = '/kaggle/working/pm'
shutil.copytree(SRC, DST, dirs_exist_ok=True)
os.chdir(DST)
print('source:', SRC, '->', DST)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)

# --- point the studio at the attached Gemma; measured signals need real logits ---
gcfg = [p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma' in p.lower()]
assert gcfg, 'gemma model not attached'
os.environ['GEMMA_MODEL_PATH'] = os.path.dirname(gcfg[0])
os.environ['GEMMA_PROVIDER'] = 'transformers'
print('gemma:', os.environ['GEMMA_MODEL_PATH'])

# Optional hosted-LLM panel keys via Kaggle user secrets (add them in the
# notebook editor: Add-ons -> Secrets). Missing secrets are fine — the panel
# simply lists itself in dry-run mode and Gemma stays the core engine.
try:
    from kaggle_secrets import UserSecretsClient
    usc = UserSecretsClient()
    for name in ('NVIDIA_API_KEY', 'OLLAMA_CLOUD_API_KEY', 'ADVISOR_LLM_API_KEY',
                 'MISTRAL_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY'):
        try:
            os.environ[name] = usc.get_secret(name)
            print('panel key loaded:', name)
        except Exception:
            pass
except Exception:
    pass

In [ ]:
# --- start the studio ---
st_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port', '8501', '--server.address', '0.0.0.0',
     '--server.headless', 'true', '--browser.gatherUsageStats', 'false'],
    stdout=open('/kaggle/working/streamlit.log', 'w'), stderr=subprocess.STDOUT)
for i in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=2)
        print(f'streamlit up after {i+1}s'); break
    except Exception:
        time.sleep(1)
else:
    print(open('/kaggle/working/streamlit.log').read()[-3000:]); raise RuntimeError('streamlit never came up')

In [ ]:
# --- cloudflared quick tunnel ---
CF = '/kaggle/working/cloudflared'
urllib.request.urlretrieve(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
os.chmod(CF, 0o755)
cf_proc = subprocess.Popen([CF, 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
deadline = time.time() + 120
buf = []
while time.time() < deadline and url is None:
    line = cf_proc.stdout.readline()
    if not line: time.sleep(0.2); continue
    buf.append(line)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0)
assert url, 'no tunnel URL: ' + ''.join(buf)[-2000:]
TOPIC = 'punchline-mesh-live-k7q2xa41'
def announce(u):
    try:
        req = urllib.request.Request(f'https://ntfy.sh/{TOPIC}', data=u.encode(),
                                     headers={'Title': 'punchline-mesh live'})
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        print('announce failed:', e)
announce(url)
print('=' * 70)
print('LIVE STUDIO URL:', url)
print('=' * 70)

In [ ]:
# --- keep-alive ~8h, re-announce hourly, restart tunnel if it drops, exit clean ---
END = time.time() + 8 * 3600
last_announce = time.time()
while time.time() < END:
    time.sleep(30)
    if st_proc.poll() is not None:
        print('streamlit died; tail:'); print(open('/kaggle/working/streamlit.log').read()[-2000:]); break
    if cf_proc.poll() is not None:
        print('tunnel died; restarting')
        cf_proc = subprocess.Popen([CF, 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        t0 = time.time(); url2 = None
        while time.time() - t0 < 120 and url2 is None:
            line = cf_proc.stdout.readline()
            m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
            if m: url2 = m.group(0)
        if url2: url = url2; announce(url); print('new URL:', url)
    if time.time() - last_announce > 3600:
        announce(url); last_announce = time.time()
print('session end; final URL was:', url)
for p in (cf_proc, st_proc):
    try: p.terminate()
    except Exception: pass